# 0. Import Libraries

In [ ]:
import os, sys
from pathlib import Path

def add_project_path(module_folder="model_implementation"):
    candidates = [
        os.path.abspath("."),
        os.path.abspath("../src"),
        os.path.abspath("src"),
    ]

    for path in candidates:
        if os.path.exists(os.path.join(path, module_folder)):
            if path not in sys.path:
                sys.path.append(path)
            return path

    raise ImportError(f"Could not find '{module_folder}' in current or parent directory")

SRC_PATH = add_project_path("model_implementation")
add_project_path("cnn")
add_project_path("rnn")
ROOT = Path(SRC_PATH).parent.resolve()
print("ROOT:", ROOT)

In [ ]:
# Jalankan cell ini hanya bila environment belum memiliki dependency.
# import subprocess
# subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow", "scikit-learn", "pandas", "matplotlib", "nltk", "pillow"])

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np

try:
    import tensorflow as tf
except Exception as exc:
    tf = None
    print("TensorFlow belum tersedia:", exc)

from cnn.utility import feature_extractor, image_paths
from common.io import load_npy

# 1. Global Variables

In [ ]:
SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 10
MAX_CAPTION_LENGTH = 34

np.random.seed(SEED)
plt.style.use("seaborn-v0_8")

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FEATURE_DIR = DATA_DIR / "features"
VOCAB_DIR = DATA_DIR / "vocab"
MODEL_DIR = ROOT / "models"
CNN_MODEL_DIR = MODEL_DIR / "cnn"
RNN_MODEL_DIR = MODEL_DIR / "rnn"
REPORT_DIR = ROOT / "reports"
TABLE_DIR = REPORT_DIR / "tables"
FIG_DIR = REPORT_DIR / "figures"

for path in [FEATURE_DIR, VOCAB_DIR, CNN_MODEL_DIR, RNN_MODEL_DIR, TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CATEGORIES = {
    "buildings": 0,
    "forest": 1,
    "glacier": 2,
    "mountain": 3,
    "sea": 4,
    "street": 5,
}
INV_CATEGORIES = {v: k for k, v in CATEGORIES.items()}

if tf is not None:
    gpu_devices = tf.config.list_physical_devices("GPU")
    if gpu_devices:
        print("GPU Detected:", gpu_devices)
        for gpu in gpu_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print("No GPU found, defaulting to CPU.")

# 2. Flickr8k Feature Extraction

In [ ]:
IMAGE_DIR = RAW_DIR / "flickr8k" / "Images"
FEATURE_PATH = FEATURE_DIR / "flickr8k_features.npy"
FEATURE_ID_PATH = FEATURE_DIR / "flickr8k_image_ids.txt"
ENCODER_PATH = CNN_MODEL_DIR / "flickr8k_inceptionv3_encoder.keras"

features = None
feature_image_ids = []

if tf is not None and IMAGE_DIR.exists():
    img_paths = image_paths(IMAGE_DIR)
    feature_image_ids = [path.stem for path in img_paths]
    if ENCODER_PATH.exists():
        encoder = tf.keras.models.load_model(ENCODER_PATH)
    else:
        encoder = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet', input_shape=(299, 299, 3), pooling='avg')
        encoder.trainable = False
        encoder.save(ENCODER_PATH)
    features = feature_extractor(img_paths, encoder, FEATURE_PATH, target_size=encoder.input_shape[1:3], batch_size=BATCH_SIZE, image_id_path=FEATURE_ID_PATH, preprocess_fn=tf.keras.applications.inception_v3.preprocess_input)
    print("images:", len(feature_image_ids))
    print("features:", features.shape)
elif FEATURE_PATH.exists() and FEATURE_ID_PATH.exists():
    features = load_npy(FEATURE_PATH).astype('float32')
    feature_image_ids = [line.strip() for line in FEATURE_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    print("Loaded existing features:", features.shape, len(feature_image_ids))
else:
    print("Feature extraction dilewati.")